In [1]:
import pandas as pd
import numpy as np
from sklearn.ensemble import HistGradientBoostingRegressor
from sklearn.model_selection import KFold
from sklearn.metrics import mean_absolute_error

In [2]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [3]:
cd drive/MyDrive/

/content/drive/MyDrive


In [4]:
train = pd.read_csv('traffic_V6.csv')
test = pd.read_csv('test_traffic_V6.csv')

print(f"학습 데이터 크기: {train.shape}")
print(f"테스트 데이터 크기: {test.shape}")

학습 데이터 크기: (249944, 108)
테스트 데이터 크기: (50000, 107)


In [5]:
TARGET = "avg_delay_minutes_next_30m"
COLS = ["ID", "layout_id", "scenario_id"]

feature_cols = [c for c in train.columns if c not in COLS + [TARGET]]
print(f"피처 수: {len(feature_cols)}")

피처 수: 104


In [6]:
from sklearn.preprocessing import LabelEncoder

for col in ["layout_type"]:
    le = LabelEncoder()
    train[col] = le.fit_transform(train[col])
    test[col] = le.transform(test[col])

In [7]:
kf = KFold(n_splits=5, shuffle=True, random_state=42)
oof_preds = np.zeros(len(train))
test_preds = np.zeros(len(test))

for fold, (tr_idx, val_idx) in enumerate(kf.split(train)):
    print(f"── Fold {fold + 1} ──")
    X_tr = train.loc[tr_idx, feature_cols]
    y_tr = train.loc[tr_idx, TARGET]
    X_val = train.loc[val_idx, feature_cols]
    y_val = train.loc[val_idx, TARGET]

    model = HistGradientBoostingRegressor(
      max_iter=500,
      learning_rate=0.05,
      max_depth=7,
      random_state=42,
      verbose=1
    )

    model.fit(X_tr, y_tr)

    val_pred = model.predict(X_val)
    fold_mae = mean_absolute_error(y_val, val_pred)
    print(f"Fold {fold+1} MAE: {fold_mae:.4f}")

    oof_preds[val_idx] = val_pred
    test_preds += model.predict(test[feature_cols]) / 5

── Fold 1 ──
Binning 0.150 GB of training data: 2.117 s
Binning 0.017 GB of validation data: 0.163 s
Fitting gradient boosted rounds:
Fit 500 trees in 41.024 s, (15500 total leaves)
Time spent computing histograms: 27.948s
Time spent finding best splits:  6.787s
Time spent applying splits:      1.723s
Time spent predicting:           0.225s
Fold 1 MAE: 8.9345
── Fold 2 ──
Binning 0.150 GB of training data: 2.232 s
Binning 0.017 GB of validation data: 0.154 s
Fitting gradient boosted rounds:
Fit 500 trees in 39.955 s, (15500 total leaves)
Time spent computing histograms: 26.764s
Time spent finding best splits:  6.783s
Time spent applying splits:      1.686s
Time spent predicting:           0.219s
Fold 2 MAE: 8.9320
── Fold 3 ──
Binning 0.150 GB of training data: 1.943 s
Binning 0.017 GB of validation data: 0.154 s
Fitting gradient boosted rounds:
Fit 500 trees in 39.449 s, (15500 total leaves)
Time spent computing histograms: 26.878s
Time spent finding best splits:  6.588s
Time spent ap

In [8]:
oof_mae = mean_absolute_error(train[TARGET], oof_preds)
print(f"OOF MAE: {oof_mae:.4f}")

OOF MAE: 8.9318


In [9]:
submission = pd.DataFrame({'ID': test['ID'], TARGET: test_preds})
submission.to_csv('submission_V27.csv', index=False)
print("submission.csv 저장 완료.")

submission.csv 저장 완료.
